https://colab.research.google.com/drive/1cZzWq5zzLk8xjUP-dMSxyHny2hm9fvQ7#scrollTo=meWGXcJdyN44

In [4]:
# !pip install langchain-core

In [2]:
# !pip install langchain-community

In [5]:
# !pip install langchain-huggingface

In [7]:
!pip install langchain-classic

In [10]:
!pip install langchain-ollama langchain-core

In [14]:
!pip install langchain-text-splitters


In [18]:
!pip install -qU langchain-text-splitters


In [22]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [8]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.5 MB/s eta 0:00:00


In [9]:
from langchain_groq import ChatGroq

In [1]:
from langchain_core.prompts import PromptTemplate
# from langchain_community.chat_models import ChatGroq

from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

from langchain_classic import hub
from langchain_ollama.llms import OllamaLLM
import ollama
from langchain_community.embeddings import OllamaEmbeddings

from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings
hf = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L12-v2'
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import GPT4AllEmbeddings

/tmp/ipykernel_5107/3853199113.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
text = ['/content/drive/MyDrive/2026/GenAI/documents/data_dictionary.txt',
        '/content/drive/MyDrive/2026/GenAI/documents/data_summary.txt',
        '/content/drive/MyDrive/2026/GenAI/documents/shap_summary.txt']

docs = [TextLoader(url).load() for url in text]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400, chunk_overlap=100
)
doc_splits = text_splitter.split_documents(docs_list)

# Add to vectorDB
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="churn-rag-chroma-2",
    embedding=hf,

)

First we will create a Business Analyst and Data Science Manager to reports from the model. This is simple agents created by Prompt defention and simple RAG.

Multiple questions are asked to the business analyst agent and then the answers for these questions are passed as context to Data Science Manger

In [ ]:
from langchain_groq import ChatGroq

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def business_analyst1(question, model, top_k=7, fetch_k=20, retriver_type="normal", search_type="similarity"):

    prompt = PromptTemplate(
        template=""" You are a business analyst for a telecom company.
        Your job is to answer question from executives on Churn.
        Answer ONLY with stats you got from the context provided.

        Context: {context}

        Question: {question}""",
        input_variables=["question", "context"],
    )

    retriever1 = vectorstore.as_retriever(search_kwargs={'k': top_k})
    docs = retriever1.invoke(question)

    document = format_docs(docs)

    llm0 = ChatGroq(
        model="llama-3.3-70b-versatile",
        groq_api_key="put_your_api"
    )

    rag_chain0 = prompt | llm0 | StrOutputParser()

    answer = rag_chain0.invoke({
        "question": question,
        "context": document
    })

    return answer



def manager(question,model,output1,output2,output3,output4):

    # Prompt
    prompt = PromptTemplate(
        template=""" You are manager of data science team for a telecom company.
                You get different reports on churn from your business analyst in your team.
                Your job is to format and proof read the report provided to you by the business analyst and create a enriched report for senior executive team within 1000 words.
                The end user might not be tech savvy, so keep the language easy to understand
                Answer should be properly formatted and user friendly and easy to understand.
                class 1 is churn.
                Use the below context for reference to make answer.

                Context: {context}

                Question: {question}""",
        input_variables=["question", "context"],
    )


    llm0 = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key="put_your_api"
)
    # Chain
    rag_chain0 = prompt | llm0 | StrOutputParser()

    answer = rag_chain0.invoke({"question": question,"context": output1 + "\n" + output2 + "\n" + output3+"\n"+output4})

    return answer

In [13]:
output1=(business_analyst1(question="Look at all features and identify the top 20 features which increases the probability of class 1 the most. The highest increase in probability will be on top",
                        model="llama3:latest",top_k=8,retriver_type='normal'))
print(output1)

Based on the context provided, here are the top 20 features that increase the probability of class 1 the most:

1. When contract is Month-to-month, it increases the probability of class 1 by 16.49%
2. When monthlycharges is within (102.75, 118.75], it increases the probability of class 1 by 15.17%
3. When totalcharges is within (18.898999999999997, 79.35], it increases the probability of class 1 by 18.09% -> This is the highest, so it should be on top
4. When totalcharges is within (5939.18, 8672.45], it increases the probability of class 1 by 3.57%
5. When internetservice is Fiber optic, it increases the probability of class 1 by 3.88%
6. When onlinesecurity is No, it increases the probability of class 1 by 3.47%
7. When streamingmovies is Yes, it increases the probability of class 1 by 1.80%
8. When streamingtv is Yes, it increases the probability of class 1 by 1.55%
9. When paperlessbilling is Yes, it increases the probability of class 1 by 1.72%
10. When onlinebackup is No, it incr

In [14]:
output2=(business_analyst1(question="Look at all features and identify the top 10 features which decreases the probability of class 1 the most. The highest decrease in probability will be on top",
                        model="llama3:latest",top_k=8,retriver_type='normal'))
print(output2)

Based on the provided context, the top 10 features that decrease the probability of class 1 the most are:

1. When tenure is within (70.0, 72.0], it decreases the probability of class 1 by 21.36%.
2. When contract is Two year, it decreases the probability of class 1 by 19.73%.
3. When monthlycharges is within (20.04, 24.85], it decreases the probability of class 1 by 11.49%.
4. When totalcharges is within (3159.83, 4445.63], it decreases the probability of class 1 by 10.19%.
5. When internetservice is DSL, it decreases the probability of class 1 by 4.10%.
6. When paymentmethod is Mailed check, it decreases the probability of class 1 by 4.60%.
7. When onlinesecurity is Yes, it decreases the probability of class 1 by 6.08%, however when onlinesecurity is No internet service, it decreases the probability of class 1 by 4.83%  and when onlinesecurity is No, it increases the probability of class 1. The overall highest decrease is 6.08%.
8. When tenure is within (61.0, 70.0], it decreases the

In [17]:
output3=(business_analyst1(question="Look at all features and identify the top 10 features importance rank in the model prediction. The lowest importance rank in the model prediction be on top",
                        model="llama3:latest",top_k=8,retriver_type='normal'))
print(output3)

Based on the provided context, the top 10 features importance rank in the model prediction are:

1. contract (importance rank: 1, mean absolute SHAP value: 1.0522)
2. tenure (importance rank: 2, mean absolute SHAP value: 0.6248)
3. monthlycharges (importance rank: 3, mean absolute SHAP value: 0.5191)
4. totalcharges (importance rank: 4, mean absolute SHAP value: 0.4363)
5. onlinesecurity (importance rank: 5, mean absolute SHAP value: 0.2614)
6. paymentmethod (importance rank: 6, mean absolute SHAP value: 0.2245)
7. techsupport (importance rank: 7, mean absolute SHAP value: 0.2001)
8. internetservice (importance rank: 8, mean absolute SHAP value: 0.1948)
9. multiplelines (importance rank: 9, mean absolute SHAP value: 0.1458)
10. paperlessbilling (importance rank: 10, mean absolute SHAP value: 0.1444)

The lowest importance rank in this list is 10, which is paperlessbilling.


In [15]:
output4=(business_analyst1(question="What are the list of different action features?",
                        model="llama3:latest",top_k=3,retriver_type='normal'))
print(output4)

The list of different action features are: 
1. PhoneService 
2. InternetService 
3. OnlineBackup 
4. OnlineSecurity 
5. DeviceProtection 
6. TechSupport 
7. Contract


In [18]:
final_report=manager(question=f"""What are the top 5 reasons for churn?\n
                             What are top 5 recommended actions to improve retention and decrease churn?""",
                             output1=output1,
                             output2=output2,
                             output3=output3,
                             output4=output4,
                             model="llama3:latest")
print(final_report)

**Churn Analysis Report**

**Introduction**
---------------

Churn is a critical issue in the telecom industry, and understanding its causes is essential to improve customer retention. This report analyzes the top reasons for churn and provides recommendations to decrease churn and improve retention.

**Top 5 Reasons for Churn**
-------------------------

Based on our analysis, the top 5 reasons for churn are:

1. **Total Charges**: When total charges are within the range of (18.898999999999997, 79.35], it increases the probability of churn by 18.09%.
2. **Contract Type**: Month-to-month contracts increase the probability of churn by 16.49%.
3. **Monthly Charges**: When monthly charges are within the range of (102.75, 118.75], it increases the probability of churn by 15.17%.
4. **Tenure**: When tenure is within the range of (-0.001, 2.0], it increases the probability of churn by 13.86%.
5. **Internet Service**: Fiber optic internet service increases the probability of churn by 3.88%.



In [19]:
response2=manager(question=f"""
                             What are top 5 recommended actions to improve retention and decrease churn specifically using action features?
                             Only use action features for this recommnedation""",
                             output1=output1,
                             output2=output2,
                             output3=output3,
                             output4=output4,
                             model="llama3:latest")
print(response2)

**Recommendations to Improve Retention and Decrease Churn**

Based on the analysis, the top 5 recommended actions to improve retention and decrease churn using action features are:

1. **Offer Online Security**: Providing online security to customers can decrease the probability of churn by 6.08%. This can be achieved by offering online security services, such as virus protection and firewall protection, to customers.
2. **Provide Tech Support**: Offering tech support to customers can decrease the probability of churn by 5.05%. This can be achieved by providing technical support services, such as phone support and online chat support, to customers.
3. **Promote Online Backup**: Promoting online backup services to customers can decrease the probability of churn. Although the exact decrease in churn probability is not specified, online backup is an action feature that can improve customer satisfaction and loyalty.
4. **Offer Device Protection**: Offering device protection services to cus

### LangGraph Agents
What are LangGraph Agents?
In the LangChain and LangGraph context, an agent is a sophisticated construct that utilizes language models (LLMs) to reason and decide on actions in an autonomous manner. Unlike traditional LangChain chains that follow a predefined sequence, agents dynamically determine their next steps based on the current situation and goals.
Building Agents with LangGraph
Define Nodes: Each node in the LangGraph represents a function or a LangChain runnable (like an LLMChain).

Connect with Edges: Define edges to dictate the flow of information and control between nodes. You'll often have conditional edges where the next node is determined based on the output of the current node.

Agent Logic: One or more nodes will typically contain the agent's decision-making logic, using an LLM to analyze the current state and choose the next action.
We will try to implement an advanced RAG system for the above Task which will be more accurate. We will design this system as LangGraph agents. We will add the Business Analysts and Manager Agents into this architecture

In [26]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser

from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser
from typing_extensions import TypedDict
from typing import List
from langchain_core.documents import Document
from langgraph.graph import END, StateGraph
from pprint import pprint

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "put_your_api"

In [29]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_groq import ChatGroq

In [30]:
# Fast LLM for grading
llm_fast = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Smart LLM for report generation
llm_smart = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [31]:
retrieval_prompt = PromptTemplate(
    template="""You are a grader assessing relevance of a retrieved document to a user question.

Here is the retrieved document:
{document}

Here is the user question:
{question}

If the document contains keywords related to the user question, grade it as relevant.

Give a binary score 'yes' or 'no'.

Return JSON format:
{{"score":"yes"}} or {{"score":"no"}}
""",
    input_variables=["question", "document"],
)

retrieval_grader = retrieval_prompt | llm_fast | JsonOutputParser()

In [32]:
buisiness_analyst_prompt = PromptTemplate(
    template="""You are a business analyst for a telecom company.

Your job is to create a report for senior executives on questions asked.

Answer ONLY with stats you got from the context provided.
DO NOT add any erroneous stats.

Context:
{context}

Question:
{question}
""",
    input_variables=["question", "context"],
)

business_analyst_rag_chain = buisiness_analyst_prompt | llm_smart | StrOutputParser()

In [33]:
manager_prompt = PromptTemplate(
    template="""You are manager of a data science team for a telecom company.

You receive multiple reports from business analysts.

Your job is to:

• Combine the reports
• Proofread them
• Create a clear executive summary

The final report must:
- be easy to understand
- be properly formatted
- avoid technical jargon

Note: class 1 represents churn.

Context:
{context}

Question:
{question}
""",
    input_variables=["question", "context"],
)

manager_rag_chain = manager_prompt | llm_smart | StrOutputParser()

In [34]:
hallucination_prompt = PromptTemplate(
    template="""You are a grader assessing whether an answer is grounded in facts.

Facts:
{documents}

Answer:
{generation}

Return a binary score 'yes' or 'no'.

Return JSON format:
{{"score":"yes"}} or {{"score":"no"}}
""",
    input_variables=["generation", "documents"],
)

hallucination_grader = hallucination_prompt | llm_fast | JsonOutputParser()

In [35]:
answer_grader_prompt = PromptTemplate(
    template="""You are grading whether an answer properly resolves a question.

Answer:
{generation}

Question:
{question}

Return a binary score 'yes' or 'no'.

Return JSON format:
{{"score":"yes"}} or {{"score":"no"}}
""",
    input_variables=["generation", "question"],
)

answer_grader = answer_grader_prompt | llm_fast | JsonOutputParser()

In [36]:
re_write_prompt = PromptTemplate(
    template="""You are a question rewriter.

Your job is to rewrite a question so it becomes optimized for vector database retrieval.

Original question:
{question}

Return ONLY the improved question.
Do not add any explanation.
""",
    input_variables=["question"],
)

question_rewriter = re_write_prompt | llm_fast | StrOutputParser()

In [37]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [39]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20}
)

In [40]:
question = "Which features increase churn probability the most?"

docs = retriever.invoke(question)

filtered_docs = []

for d in docs:
    score = retrieval_grader.invoke({
        "question": question,
        "document": d.page_content
    })

    if score["score"] == "yes":
        filtered_docs.append(d)

context = format_docs(filtered_docs)

# Generate analyst report
answer = business_analyst_rag_chain.invoke({
    "question": question,
    "context": context
})

print(answer)

Based on the provided context, the features that increase churn probability the most are:

1. Contract: Month-to-month, which increases the probability of class 1 by 16.49%.
2. TotalCharges: When totalcharges is within (18.898999999999997, 79.35], it increases the probability of class 1 by 18.09%.
3. MonthlyCharges: When monthlycharges is within (102.75, 118.75], it increases the probability of class 1 by 15.17%.
4. Tenure: When tenure is within (-0.001, 2.0], it increases the probability of class 1 by 13.86%.
5. PaymentMethod: Electronic check, which increases the probability of class 1 by 4.13%.


In [41]:
class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM generation
        documents: list of documents
        report: FInal report from Manager LLM
    """
    question : str
    generation : str
    documents : List[str]
    report : str

In [42]:
### Nodes
def retrieve(state):
    """
    Retrieve documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, documents, that contains retrieved documents
    """
    print("---RETRIEVE---")
    question = state["question"]

    # Retrieval
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}

def analyst_generate(state):
    """
    Generate answer

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]
    context=format_docs(documents)

    # RAG generation
    generation = business_analyst_rag_chain.invoke({"context": context, "question": question})
    return {"documents": documents, "question": question, "generation": generation}

def grade_documents(state):
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with only filtered relevant documents
    """

    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]

    # Score each doc
    filtered_docs = []
    for d in documents:
        score = retrieval_grader.invoke({"question": question, "document": d.page_content})
        grade = score['score']
        if grade == "yes":
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(d)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            continue
    return {"documents": filtered_docs, "question": question}

def transform_query(state):
    """
    Transform the query to produce a better question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates question key with a re-phrased question
    """

    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]

    # Re-write question
    better_question = question_rewriter.invoke({"question": question})
    return {"documents": documents, "question": better_question}


def decide_to_generate(state):
    """
    Determines whether to generate an answer, or re-generate a question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Binary decision for next node to call
    """

    print("---ASSESS GRADED DOCUMENTS---")
    question = state["question"]
    filtered_documents = state["documents"]

    if not filtered_documents:
        # All documents have been filtered check_relevance
        # We will re-generate a new query
        print("---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---")
        return "transform_query"
    else:
        # We have relevant documents, so generate answer
        print("---DECISION: GENERATE---")
        return "analyst_generate"

def grade_generation_v_documents_and_question(state):
    """
    Determines whether the generation is grounded in the document and answers question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Decision for next node to call
    """

    print("---CHECK HALLUCINATIONS---")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]

    score = hallucination_grader.invoke({"documents": format_docs(documents), "generation": generation})
    grade = score['score']

    # Check hallucination
    if grade == "yes":
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")
        # Check question-answering
        print("---GRADE GENERATION vs QUESTION---")
        score = answer_grader.invoke({"question": question,"generation": generation})
        grade = score['score']
        if grade == "yes":
            print("---DECISION: GENERATION ADDRESSES QUESTION---")
            return "useful"
        else:
            print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION---")
            return "not useful"
    else:
        pprint("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"

def manager_generate(state):
    """
    Generate answer

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    print("---GENERATE---")
    question = state["question"]
    context=state["generation"]

    # RAG generation
    report = manager_rag_chain.invoke({"context": context, "question": question})
    return {"question": question, "report": report}

In [43]:
## Create the Graph Agent

workflow = StateGraph(GraphState)

# Define the nodes
workflow.add_node("retrieve", retrieve) # retrieve
workflow.add_node("grade_documents", grade_documents) # grade documents
workflow.add_node("analyst_generate", analyst_generate) # generatae
workflow.add_node("transform_query", transform_query) # transform_query
workflow.add_node("manager_generate", manager_generate) # manager_query

# Build graph
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "analyst_generate": "analyst_generate",
    },
)
workflow.add_edge("transform_query", "retrieve")
workflow.add_conditional_edges(
    "analyst_generate",
    grade_generation_v_documents_and_question,
    {
        "not supported": "analyst_generate",
        "useful": "manager_generate",
        "not useful": "transform_query",
    },
)
workflow.add_edge("manager_generate",END)

# Compile
app = workflow.compile()

In [44]:
##Test the Agent with a Question

# Run
inputs = {"question": "What are the top reasons which increases the probability of class 1?"}
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
    pprint("\n---\n")

# Final generation
pprint(value["report"])

---RETRIEVE---
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
"Node 'grade_documents':"
'\n---\n'
---GENERATE---
---CHECK HALLUCINATIONS---
---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
---GRADE GENERATION vs QUESTION---
---DECISION: GENERATION ADDRESSES QUESTION---
"Node 'analyst_generate':"
'\n---\n'
---GENERATE---
"Node 'manager_generate':"
'\n---\n'
('**Executive Summary: Top Rea

In [45]:
inputs = {"question": "What are 5 recommended actions to increase retention or to reduce churn?"}
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
    pprint("\n---\n")

# Final generation
pprint(value["report"])

---RETRIEVE---
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---


OutputParserException: Invalid json output: The document contains keywords related to the user question, such as "probability of class 1" which can be interpreted as a measure of retention or churn. The document also mentions specific actions that can increase or decrease the probability of class 1, which is relevant to the user question.

{"score":"yes"}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 